### ~

In [ ]:

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

#
from pathlib import Path
import shutil
import subprocess
import builtins



ROOT = Path.cwd()

def here():
    return ROOT

def make_dir(path):
    p = ROOT / path
    p.mkdir(parents=True, exist_ok=True)
    return p

def make_file(path, content=""):
    p = ROOT / path
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
    return p

def remove_path(path):
    p = ROOT / path
    if p.is_dir():
        shutil.rmtree(p)
    elif p.exists():
        p.unlink()
    return p

def run_cmd(*cmd):
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.returncode
#
import os
from pathlib import Path
import shutil
import subprocess


class WorkspaceBase:
    @staticmethod
    def get_notebook_name():
        notebook_files = [f for f in os.listdir('.') if f.endswith('.ipynb')]
        if notebook_files:
            thienotebooktitleworkspace = os.path.splitext(notebook_files[0])[0]
        else:
            thienotebooktitleworkspace = "Untitled Notebook"
        return thienotebooktitleworkspace

    def get_root(self):
        return self.root

    def make_folder(self, name):
        folder = self.root / name
        folder.mkdir(parents=True, exist_ok=True)
        return folder

    def write_text(self, path, text):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
        return path

    def read_text(self, path):
        return Path(path).read_text(encoding="utf-8")

    def run_cmd(self, *args):
        result = subprocess.run(args, capture_output=True, text=True)
        print(result.stdout)
        if result.stderr:
            print("Error:", result.stderr)

    def __init__(self, root=None):
        if root is None:
            root = f"{self.get_notebook_name()}_workspace"
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

class Problem_(WorkspaceBase):
    def __init__(self, name, files=None, root=None):
        super().__init__(root=root)
        self.name = name
        self.files = files or {}
        self.folder = self.make_folder(self.safe_name())

    def safe_name(self):
        return self.name.strip().lower().replace(" ", "_")

    def add_file(self, relative_path, content):
        if content and content[0] == '\n':
            content = content[1:]
        self.files[str(relative_path)] = content

    def get_file(self, relative_path):
        return self.files.get(str(relative_path))

    def remove_file(self, relative_path):
        self.files.pop(str(relative_path), None)

    def save(self, clear_first=False):
        if clear_first:
            self.reset_folder(self.folder)
        for relative_path, content in self.files.items():
            full_path = self.folder / relative_path
            self.write_text(full_path, content)

    def run(self, command, cwd=None, inputs=None):
        """Run a shell command. Pass inputs as a list of strings for stdin."""
        if cwd is None:
            cwd = self.folder
        stdin_text = "\n".join(str(v) for v in inputs) + "\n" if inputs else None
        result = subprocess.run(command, cwd=cwd, capture_output=True, text=True,
                                shell=True, input=stdin_text)
        print(result.stdout)
        if result.stderr:
            print("Error:", result.stderr)

    def reset_folder(self, folder):
        folder = Path(folder)
        if folder.exists():
            shutil.rmtree(folder)
        folder.mkdir(parents=True, exist_ok=True)
        return folder

    def load(self):
        loaded = {}
        for relative_path in self.list_all_files(self.folder):
            full_path = self.folder / relative_path
            loaded[relative_path] = self.read_text(full_path)
        self.files = loaded
        return loaded

    def summary(self):
        return {
            "name": self.name,
            "folder": str(self.folder),
            "file_count": len(self.files),
            "files": sorted(self.files.keys())
        }

    def __repr__(self):
        return f"Problem(name={self.name!r}, files={len(self.files)}, folder={str(self.folder)!r})"

class Problem(Problem_):
    def __init__(self, name, files=None, root=None):
        super().__init__(name, files, root)
        self.reset_folder(self.folder)

    def runpy(self, command, cwd=None, inputs=None):
        """Run a python file. Pass inputs as a list of strings for stdin.
        Example: _.runpy("main.py", inputs=["8", "3"])
        """
        print(f"Ran command: python {command} in {cwd or self.folder}")
        self.run(f"python {command}", cwd, inputs=inputs)

    def create(self, relative_path, content):
        self.add_file(relative_path, content)
        self.save()
        print(f"Created file: {relative_path} with content:\n{content}")

    def create2(self, relative_path, content, inputs=None):
        self.create(relative_path, content)
        self.runpy(relative_path, cwd=self.folder, inputs=inputs)

    def answer(self, content):
        self.add_file("ANSWER.md", content)
        self.save()
        print(f"Saved answer to ANSWER.md:\n{content}")

    def q(self, text):
        """Display the question/prompt text. Strips leading/trailing whitespace."""
        print(text.strip())
        return self

    def parse(self, text):
        """
        Mini-DSL to create files and run commands from one string.

        Syntax:
            Classic:
                F:
                <file content goes here>
                F:: filename.py          <- saves the content block to filename.py

            Reversed:
                F:: filename.py
                <file content goes here>
                F:                       <- closes and saves

            End-only:
                <file content goes here>
                F:: filename.py

            Alternate filename marker:
                FF: filename.py

            Run commands:
                R: main.py               <- runs main.py with no input
                R: main.py ["Eve", "5"]  <- runs main.py with those inputs

        Example:
            _.parse('''
            F:
            def greet(): return "hi"
            F:: greet.py

            F:
            import greet
            print(greet.greet())
            F:: main.py

            R: main.py
            ''')
        """
        import ast as _ast

        def _filename_from_marker(stripped_line):
            if stripped_line.startswith('F::'):
                return stripped_line[3:].strip()
            if stripped_line.startswith('FF:'):
                return stripped_line[3:].strip()
            if stripped_line.startswith('FILE::'):
                return stripped_line[6:].strip()
            return None

        lines = text.split('\n')
        collecting = False
        buffer = []
        current_filename = None
        mode = None  # "normal" (F: ... F::name) or "reversed" (F::name ... F:)
        loose_buffer = []

        for line in lines:
            stripped = line.strip()

            marker_filename = _filename_from_marker(stripped)

            if stripped == 'F:':
                if collecting and mode == "reversed" and current_filename:
                    self.create(current_filename, '\n'.join(buffer))
                    collecting = False
                    buffer = []
                    current_filename = None
                    mode = None
                else:
                    collecting = True
                    buffer = []
                    current_filename = None
                    mode = "normal"

            elif marker_filename is not None:
                if collecting and mode == "normal":
                    self.create(marker_filename, '\n'.join(buffer))
                    collecting = False
                    buffer = []
                    current_filename = None
                    mode = None
                elif not collecting:
                    if loose_buffer:
                        self.create(marker_filename, '\n'.join(loose_buffer))
                        loose_buffer = []
                    else:
                        collecting = True
                        buffer = []
                        current_filename = marker_filename
                        mode = "reversed"
                else:
                    self.create(marker_filename, '\n'.join(buffer))
                    collecting = False
                    buffer = []
                    current_filename = None
                    mode = None

            elif stripped.startswith('R:'):
                rest = stripped[2:].strip()
                # Split off optional inputs list at the end, e.g.  main.py ["a","b"]
                inputs = None
                bracket_idx = rest.find('[')
                if bracket_idx != -1:
                    filename = rest[:bracket_idx].strip()
                    try:
                        inputs = _ast.literal_eval(rest[bracket_idx:])
                        inputs = [str(x) for x in inputs]
                    except Exception:
                        inputs = None
                else:
                    filename = rest
                if filename:
                    self.runpy(filename, inputs=inputs)

            elif collecting:
                buffer.append(line)

            elif stripped:
                loose_buffer.append(line)

        if collecting and mode == "reversed" and current_filename:
            self.create(current_filename, '\n'.join(buffer))

        return self

#
def setin(*inputs):
    """
    A helper function to set test inputs for the input() function.
    Usage:
    setin("input1", "input2", "input3")
    This will set up the input() function to return "input1" on the first call,
    "input2" on the second call, and so on.
    To reset to normal input behavior, call setin() with no arguments or None:
    setin()
    """
    if inputs:
        input_iter = iter(inputs)
        def mock_input(prompt=""):
            try:
                value = next(input_iter)
                print(f"{prompt}{value}")
                return value
            except StopIteration:
                raise EOFError("No more inputs for testing")
        builtins.input = mock_input
    else:
        builtins.input = builtins._original_input_backup
              
#


This is a utility module. Please import and use the functions as needed.


### ~